# IOAI — 2026 Local Mock Parkinson (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-parkinson.zip', '/tmp/park.zip')
    with zipfile.ZipFile('/tmp/park.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
# catboost 는 ioai-venv 에 없어 sklearn HistGradientBoosting 으로 대체(부스팅, 동등 성능)
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
root_path = "data"
seed = 42

# Data

In [ ]:
def clean_df(df: pd.DataFrame):
    df = df.drop(["PatientID", "Diagnosis", "Ethnicity", "EducationLevel"], axis=1, errors="ignore")
    df = df.select_dtypes(exclude=['object'])

    df["CardiometabolicRiskScore"] = (
        df["Hypertension"] + df["Diabetes"] + (df["BMI"] > 30)
    )

    df["LifestyleRiskIndex"] = (
        df["Smoking"] + (df["AlcoholConsumption"] > 2) + (df["PhysicalActivity"] < 1)
    )

    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        dummy = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummy], axis=1)
        df = df.drop(col, axis=1)

    return df

In [ ]:
train_df_ = pd.read_csv(f"{root_path}/train.csv")
train_df = clean_df(train_df_)
train_df_.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_df, train_df_["Diagnosis"], test_size=0.2, random_state=seed)

## EDA

In [ ]:
train_df_.isna().sum().sum()

In [ ]:
train_df_.describe()

In [ ]:
train_df.select_dtypes(exclude='object').corrwith(train_df_["Diagnosis"]).sort_values(ascending=False)

In [ ]:
train_df_.select_dtypes(include=['object']).head(2)

In [ ]:
sns.histplot(x="DoctorInCharge", hue="Diagnosis", data=train_df_)

# Model

In [ ]:
def evaluate(model):
    cv = cross_val_score(
        model, X_train, y_train, scoring="roc_auc", cv=3, n_jobs=-1
    )
    cv = cv.mean() - cv.std()

    model.fit(X_train, y_train)
    preds = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, preds)
    return score, cv

In [ ]:
rf = RandomForestClassifier(n_estimators=500, max_depth=10, random_state=seed)
evaluate(rf)

In [ ]:
cb = HistGradientBoostingClassifier(random_state=seed)

evaluate(cb)

In [ ]:
clf = cb
clf.fit(train_df, train_df_["Diagnosis"])

# Submission

In [ ]:
test_df_ = pd.read_csv(f"{root_path}/test.csv")
test_df = clean_df(test_df_)

In [ ]:
subtask1 = test_df["CardiometabolicRiskScore"]
subtask2 = test_df["LifestyleRiskIndex"]
subtask3 = clf.predict_proba(test_df)[:, 1]

In [ ]:
def build_df(sid, answers):
    return pd.DataFrame({"PatientID": test_df_["PatientID"], "subtaskID": sid, "Answer": answers})

subtasks = [
    ("Task1", subtask1),
    ("Task2", subtask2),
    ("Task3", subtask3)
]

submission_df = pd.concat([build_df(sid, ans) for sid, ans in subtasks])

In [ ]:
submission_df.head()

In [ ]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)